# From the Dengue Exercise to ARGO - Lane A (the real experience: you prompt, the agent builds)

**SISMID 2026 - Day 2, 1:30.** You drive a coding agent (Codex, Claude Code, or Antigravity
CLI) to grow the static dengue model into ARGO, and you keep the judgment. For each step:
read the goal, paste the **prompt**, run the code the agent writes, and check the RMSE.

> Each prompt produces roughly the matching cell in the **Lane B** notebook. Not set up
> with an agent yet? Use Lane B. Data: `MX_Dengue_trends.csv`.


## Step 0: load and recap the static model

> *Load MX_Dengue_trends.csv (Dengue CDC = monthly cases; dengue, sintomas de dengue,*
> *mosquito, dengue sintomas = search). Refit the Part 1 static model: least squares of*
> *Dengue CDC on 'dengue' over the first 36 months, predict 2007-2011, and report the RMSE.*


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

CANDIDATES = ['../data/MX_Dengue_trends.csv', 'data/MX_Dengue_trends.csv', './MX_Dengue_trends.csv']
path = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
df = pd.read_csv(path)
df['Date'] = pd.to_datetime(df['Date'])

TARGET = 'Dengue CDC'
df_train = df.iloc[:36]
df_valid = df.iloc[36:].copy()

model_static = LinearRegression().fit(df_train[['dengue']], df_train[TARGET])
pred_static = model_static.predict(df_valid[['dengue']])
rmse_static = np.sqrt(np.mean((df_valid[TARGET].to_numpy() - pred_static) ** 2))

print(f'Static RMSE (2007-2011): {rmse_static:.1f}')


## Step 1: dynamic (sliding-window) training

> *Write a sliding-window dynamic model: for each month from 2007 on, fit least squares on*
> *the previous 36 months and predict the next month. Do it with just 'dengue' first. Plot*
> *dynamic vs static vs actual and report both RMSEs. Which is better?*


In [ ]:
def dynamic_predict(frame, features, window=36):
    features = [features] if isinstance(features, str) else features
    predictions = []
    for month in range(window, len(frame)):
        history = frame.iloc[month - window:month].dropna(subset=features + [TARGET])
        model = LinearRegression().fit(history[features], history[TARGET])
        predictions.append(model.predict(frame.iloc[[month]][features])[0])
    return np.asarray(predictions)

pred_dynamic = dynamic_predict(df, 'dengue')
rmse = lambda actual, prediction: np.sqrt(np.mean((np.asarray(actual) - prediction) ** 2))

print(f'Static RMSE:  {rmse(df_valid[TARGET], pred_static):.1f}')
print(f'Dynamic RMSE: {rmse(df_valid[TARGET], pred_dynamic):.1f}')

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(df_valid['Date'], df_valid[TARGET], label='Actual', color='black')
plt.plot(df_valid['Date'], pred_static, label='Static')
plt.plot(df_valid['Date'], pred_dynamic, label='Dynamic (36-month window)')
plt.ylabel('Monthly dengue cases')
plt.title('Static vs dynamic dengue predictions')
plt.legend()
plt.tight_layout()
plt.show()


## Step 2: add all four search terms

> *Extend the dynamic model to use all four search terms as predictors. Recompute the RMSE*
> *and compare to the one-term dynamic model.*


In [ ]:
SEARCH = ['dengue', 'sintomas de dengue', 'mosquito', 'dengue sintomas']
pred_dynamic_all = dynamic_predict(df, SEARCH)

print(f'Dynamic, one term RMSE:  {rmse(df_valid[TARGET], pred_dynamic):.1f}')
print(f'Dynamic, all terms RMSE: {rmse(df_valid[TARGET], pred_dynamic_all):.1f}')

plt.figure(figsize=(10, 4))
plt.plot(df_valid['Date'], df_valid[TARGET], label='Actual', color='black', linewidth=2)
plt.plot(df_valid['Date'], pred_dynamic, label='Dynamic (one term)')
plt.plot(df_valid['Date'], pred_dynamic_all, label='Dynamic (all terms)')
plt.ylabel('Monthly dengue cases')
plt.title('Dynamic dengue models: one search term vs all terms')
plt.legend()
plt.tight_layout()
plt.show()


## Step 3: add autoregression = ARGO

> *Add autoregression: include last month's and the month-before's Dengue CDC (ar1, ar2)*
> *as extra features (shift the case column; the first two rows will be NaN). Refit the*
> *sliding window with search terms + ar1 + ar2. This is ARGO. Compare its RMSE to the rest.*


In [ ]:
df_argo = df.copy()
df_argo['ar1'] = df_argo[TARGET].shift(1)
df_argo['ar2'] = df_argo[TARGET].shift(2)
ARGO_FEATURES = SEARCH + ['ar1', 'ar2']

pred_argo = dynamic_predict(df_argo, ARGO_FEATURES)

print(f'Dynamic, all terms RMSE: {rmse(df_valid[TARGET], pred_dynamic_all):.1f}')
print(f'ARGO RMSE:               {rmse(df_valid[TARGET], pred_argo):.1f}')

plt.figure(figsize=(10, 4))
plt.plot(df_valid['Date'], df_valid[TARGET], label='Actual', color='black', linewidth=2)
plt.plot(df_valid['Date'], pred_dynamic_all, label='Dynamic (all search terms)')
plt.plot(df_valid['Date'], pred_argo, label='ARGO (search + AR lags)')
plt.ylabel('Monthly dengue cases')
plt.title('ARGO compared with the dynamic search model')
plt.legend()
plt.tight_layout()
plt.show()


## Step 4: compare everything

> *Print a table of out-of-sample RMSE for static, dynamic-1-term, dynamic-all-terms, and*
> *ARGO, and overlay all four predictions against actual cases.*


In [ ]:
models = {
    'Static (one term)': pred_static,
    'Dynamic (one term)': pred_dynamic,
    'Dynamic (all terms)': pred_dynamic_all,
    'ARGO (+ AR2)': pred_argo,
}

for name, prediction in models.items():
    print(f'{name:22s} RMSE: {rmse(df_valid[TARGET], prediction):.1f}')

plt.figure(figsize=(10, 4))
plt.plot(df_valid['Date'], df_valid[TARGET], label='Actual', color='black', linewidth=2)
for name, prediction in models.items():
    plt.plot(df_valid['Date'], prediction, label=name)
plt.ylabel('Monthly dengue cases')
plt.title('From static regression to ARGO')
plt.legend()
plt.tight_layout()
plt.show()


## Stretch: full ARGO with Lasso

> *Refit the ARGO features with a Lasso (L1) model inside the sliding window (standardize*
> *the features first). Compare its RMSE to plain least-squares ARGO. Note which terms Lasso*
> *keeps.*


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def dynamic_predict_lasso(frame, features, window=36):
    predictions = []
    for month in range(window, len(frame)):
        history = frame.iloc[month - window:month].dropna(subset=features + [TARGET])
        model = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=50_000))
        model.fit(history[features], history[TARGET])
        predictions.append(model.predict(frame.iloc[[month]][features])[0])
    return np.asarray(predictions)

pred_lasso = dynamic_predict_lasso(df_argo, ARGO_FEATURES)
print(f'Least-squares ARGO RMSE: {rmse(df_valid[TARGET], pred_argo):.1f}')
print(f'Lasso ARGO RMSE:         {rmse(df_valid[TARGET], pred_lasso):.1f}')


## ARGONet discussion

> *Explain how you would extend this to ARGONet: if I had neighboring states or counties,*
> *how would I add a neighbor's lagged signal to the ARGO features, how would I pick which*
> *neighbor, and when would it help? (This dataset is one country, so answer conceptually.)*

**Reflection:** notice how little you typed. You described each upgrade; the agent wrote the
regression. The judgment (is the RMSE really lower, out of sample?) stayed with you.
